# Data exploration and preparation

This notebook loads and validates **RecipeData.csv** and **BigBasketProducts.csv**, computes dataset statistics, defines the food product subset, and creates the **evaluation set** (50 recipes) for mAP@k.

In [ ]:
import sys
from pathlib import Path

# Project root: directory containing dataset/
cwd = Path.cwd()
ROOT = cwd if (cwd / "dataset").exists() else cwd.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.load_recipes import load_recipes, parse_cleaned_ingredients, RECIPE_NAME_COL, CLEANED_INGREDIENTS_COL
from src.load_products import load_products, load_food_categories, filter_food_products

## 1. Load recipe data

In [2]:
df_recipes = load_recipes()
print("Shape:", df_recipes.shape)
print("Columns:", list(df_recipes.columns))
df_recipes.head(2)

Shape: (5938, 9)
Columns: ['TranslatedRecipeName', 'TranslatedIngredients', 'TotalTimeInMins', 'Cuisine', 'TranslatedInstructions', 'URL', 'Cleaned-Ingredients', 'image-url', 'Ingredient-count']


,TranslatedRecipeName,TranslatedIngredients,TotalTimeInMins,Cuisine,TranslatedInstructions,URL,Cleaned-Ingredients,image-url,Ingredient-count
0,Masala Karela Recipe,"1 tablespoon Red Chilli powder,3 tablespoon Gr...",45,Indian,"To begin making the Masala Karela Recipe,de-se...",https://www.archanaskitchen.com/masala-karela-...,"salt,amchur (dry mango powder),karela (bitter ...",https://www.archanaskitchen.com/images/archana...,10
1,Spicy Tomato Rice (Recipe),"2 teaspoon cashew - or peanuts, 1/2 Teaspoon ...",15,South Indian Recipes,"To make tomato puliogere, first cut the tomato...",https://www.archanaskitchen.com/spicy-tomato-r...,"tomato,salt,chickpea lentils,green chilli,rice...",https://www.archanaskitchen.com/images/archana...,12


## 2. Recipe statistics

In [3]:
# Unique ingredients from Cleaned-Ingredients
all_ingredients = []
for s in df_recipes[CLEANED_INGREDIENTS_COL].dropna():
    all_ingredients.extend(parse_cleaned_ingredients(s))
unique_ingredients = sorted(set(all_ingredients))
print("Total recipe rows:", len(df_recipes))
print("Unique ingredient names (from Cleaned-Ingredients):", len(unique_ingredients))
print("Sample ingredients:", unique_ingredients[:20])

Total recipe rows: 5938
Unique ingredient names (from Cleaned-Ingredients): 5024
Sample ingredients: ['(', '( flavour )', '( reduce )', '( than )', '(amul', '(for ganache)', '(or agave syrup)', '(or jaggery sugar)', '(or milk', '(or nondairy', '(sm scoops)', '(store bought)', ')', ') sour', 'aachi fish masala', 'aam papad (sun dried mango)', 'aamras (mango puree)', 'aar maach (fish) (rohu katla)', 'aar maach (fish) rohu katla fish (cut steaks)', 'achari masala']


In [4]:
# Cuisine and ingredient count distribution
print("Cuisine value counts (top 15):")
print(df_recipes["Cuisine"].value_counts().head(15))
print("\nIngredient-count (per recipe) - describe:")
print(df_recipes["Ingredient-count"].describe())

Cuisine value counts (top 15):
Continental              952
Indian                   927
North Indian Recipes     763
South Indian Recipes     558
Italian Recipes          231
Maharashtrian Recipes    152
Bengali Recipes          150
Karnataka                138
Tamil Nadu               133
Kerala Recipes           133
Fusion                   130
Mexican                  116
Andhra                   109
Rajasthani                99
Gujarati Recipes﻿         96
Name: Cuisine, dtype: int64

Ingredient-count (per recipe) - describe:
count    5938.000000
mean       11.826878
std         4.368183
min         3.000000
25%         9.000000
50%        11.000000
75%        14.000000
max        34.000000
Name: Ingredient-count, dtype: float64


## 3. Load product data

In [5]:
df_products = load_products()
print("Shape:", df_products.shape)
print("Columns:", list(df_products.columns))
print("\nCategory value counts (top 15):")
print(df_products["category"].value_counts().head(15))

Shape: (27555, 10)
Columns: ['index', 'product', 'category', 'sub_category', 'brand', 'sale_price', 'market_price', 'type', 'rating', 'description']

Category value counts (top 15):
Beauty & Hygiene            7867
Gourmet & World Food        4690
Kitchen, Garden & Pets      3580
Snacks & Branded Foods      2814
Foodgrains, Oil & Masala    2676
Cleaning & Household        2675
Beverages                    885
Bakery, Cakes & Dairy        851
Baby Care                    610
Fruits & Vegetables          557
Eggs, Meat & Fish            350
Name: category, dtype: int64


## 4. Food category filter

In [6]:
food_categories = load_food_categories()
print("Food categories from config:", food_categories)

df_food = filter_food_products(df_products)
print("\nProducts (all):", len(df_products))
print("Products (food only):", len(df_food))
print("\nFood category distribution:")
print(df_food["category"].value_counts())

Food categories from config: {'Fruits & Vegetables', 'Beverages', 'Foodgrains, Oil & Masala', 'Eggs, Meat & Fish', 'Bakery, Cakes & Dairy', 'Snacks & Branded Foods', 'Gourmet & World Food'}

Products (all): 27555
Products (food only): 12823

Food category distribution:
Gourmet & World Food        4690
Snacks & Branded Foods      2814
Foodgrains, Oil & Masala    2676
Beverages                    885
Bakery, Cakes & Dairy        851
Fruits & Vegetables          557
Eggs, Meat & Fish            350
Name: category, dtype: int64


In [7]:
# Save food product subset for next phase (BM25 index)
out_dir = ROOT / "output"
out_dir.mkdir(parents=True, exist_ok=True)
food_path = out_dir / "products_food.csv"
df_food.to_csv(food_path, index=False)
print(f"Saved food products to {food_path}")

Saved food products to /Users/varunssharma/Downloads/zz Recipe to Cart - Research/output/products_food.csv


## 5. Evaluation set (50 recipes for mAP@k)

In [8]:
# Sample 50 recipes (stratified by Cuisine when possible, else random)
n_eval = 50
cuisines = df_recipes["Cuisine"].dropna().unique()
try:
    if len(cuisines) >= 3:
        per_cuisine = max(1, n_eval // min(len(cuisines), 10))
        eval_df = df_recipes.groupby("Cuisine", group_keys=False).apply(
            lambda g: g.sample(n=min(len(g), per_cuisine), random_state=42)
        )
        if len(eval_df) > n_eval:
            eval_df = eval_df.sample(n=n_eval, random_state=42)
        elif len(eval_df) < n_eval:
            extra = df_recipes.drop(eval_df.index).sample(n=n_eval - len(eval_df), random_state=43)
            eval_df = pd.concat([eval_df, extra])
    else:
        eval_df = df_recipes.sample(n=min(n_eval, len(df_recipes)), random_state=42)
except Exception:
    eval_df = df_recipes.sample(n=min(n_eval, len(df_recipes)), random_state=42)
eval_df = eval_df.head(n_eval)

eval_export = eval_df[[RECIPE_NAME_COL, CLEANED_INGREDIENTS_COL]].copy()
eval_export.columns = ["recipe_name", "cleaned_ingredients"]
eval_export.insert(0, "eval_id", range(1, len(eval_export) + 1))
eval_path = out_dir / "evaluation_set_50_recipes.csv"
eval_export.to_csv(eval_path, index=False)
print(f"Saved evaluation set ({len(eval_export)} recipes) to {eval_path}")
eval_export.head(10)

Saved evaluation set (50 recipes) to /Users/varunssharma/Downloads/zz Recipe to Cart - Research/output/evaluation_set_50_recipes.csv


,eval_id,recipe_name,cleaned_ingredients
1729,1,Spinach Enchilada Recipe,"tomato,lemon,cumin powder (jeera),sour,coriand..."
4557,2,Greek Style Grilled Pita Pizza Recipe,"tomato,hummus,tme leaves dried,black peppercor..."
2820,3,Burmese Samosa Soup Recipe,"spring onion greens,kashmiri red chilli powder..."
4370,4,Struffoli Recipe,"salt,orange,curd,sugar,vanilla,lemon,lemon,flo..."
1808,5,Goan Pumpkin Sabzi Recipe,"salt,parangikai pumpkin,coconut,red chilli pow..."
218,6,Sepu Vadi Recipe (Himachali Split Urad Dal Dum...,"cumin powder (jeera),dill leaves,asafoetida (h..."
2650,7,Bengali Style Kosha Mangsho Recipe- Spicy Mutt...,"salt,coriander (dhania) leaves,mustard oil,car..."
973,8,Jerk Chicken With Rice And Peas Pilaf Recipe,"basmati rice,sunflower oil,bay leaf (tej patta..."
2644,9,Shahi Vegetable Pulao Recipe,"cashew nuts,basmati rice,salt,cardamom (elaich..."
4080,10,Rava Rotti Recipe (Karnataka Style Semolina Fl...,"salt,curd buttermilk,green chilli,sunflower oi..."


## 6. Summary

In [9]:
print("Deliverables:")
print("  - Recipe data: ", df_recipes.shape[0], "recipes, ", len(unique_ingredients), "unique ingredients")
print("  - Product data: ", df_products.shape[0], "products; ", len(df_food), "food products")
print("  - Food categories: ", len(food_categories))
print("  - Evaluation set: ", len(eval_export), "recipes at", eval_path)

Deliverables:
  - Recipe data:  5938 recipes,  5024 unique ingredients
  - Product data:  27555 products;  12823 food products
  - Food categories:  7
  - Evaluation set:  50 recipes at /Users/varunssharma/Downloads/zz Recipe to Cart - Research/output/evaluation_set_50_recipes.csv
